# Drug-Food Interaction Recommender - All In One Notebook

Notebook ini berisi alur lengkap yang sederhana:

1. Load data makanan, bahan, dan interaksi obat-makanan
2. Perbaikan data sederhana
3. Feature engineering dari bahan makanan
4. Training model **Extra Trees Regressor**
5. Evaluasi dengan MAE dan Risk Accuracy
6. Simpan model ke `models/drug_interaction_tree_model.pkl`
7. Demo rekomendasi makanan aman

Metode yang dipakai adalah **Extra Trees Regressor**, yaitu ensemble dari banyak Decision Tree. Model ini lebih cocok untuk dataset kecil dan berbasis rule/ingredient seperti project ini.

In [1]:
# ============================================================
# 1. Import Library dan Setup Path
# ============================================================

from pathlib import Path
import json
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

INTERACTIONS_PATH = DATA_DIR / "drug_food_interactions.csv"
FOOD_KB_PATH = DATA_DIR / "food_to_ingredient_kb.json"
REVERSE_KB_PATH = DATA_DIR / "ingredient_to_food_kb.json"
MODEL_OUTPUT_PATH = MODEL_DIR / "drug_interaction_tree_model.pkl"

print("Project root:", PROJECT_ROOT)
print("Data dir    :", DATA_DIR)
print("Model output:", MODEL_OUTPUT_PATH)

Project root: d:\tugas\coding camp 2026\capstone project\ai engineer
Data dir    : d:\tugas\coding camp 2026\capstone project\ai engineer\data
Model output: d:\tugas\coding camp 2026\capstone project\ai engineer\models\drug_interaction_tree_model.pkl


## 2. Load Data

Data utama:

- `food_to_ingredient_kb.json`: daftar bahan setiap kelas makanan
- `drug_food_interactions.csv`: label severity interaksi makanan dan kategori obat

In [2]:
# Load ingredient knowledge base
with open(FOOD_KB_PATH, "r", encoding="utf-8") as f:
    food_kb = json.load(f)

food_to_ingredients = food_kb["food_to_ingredients"]

# Load ground truth interaction dataset
df = pd.read_csv(INTERACTIONS_PATH)

print("Jumlah makanan       :", len(food_to_ingredients))
print("Jumlah data interaksi:", len(df))
print("Jumlah kategori obat :", df["drug_category"].nunique())

display(df.head())

Jumlah makanan       : 61
Jumlah data interaksi: 854
Jumlah kategori obat : 14


,food_class,drug_category,has_interaction,severity,interaction_type,mechanism,matched_ingredients,source
0,apel,antikoagulan,0,0,NaN,NaN,NaN,LLM-assisted curation (Claude) + pharmacology ...
1,apel,antidiabetes,1,5,pharmacodynamic,"Kandungan gula tinggi (kental manis, madu, hon...",kental manis|madu|honey,LLM-assisted curation (Claude) + pharmacology ...
2,apel,ace_arb,0,0,NaN,NaN,NaN,LLM-assisted curation (Claude) + pharmacology ...
3,apel,ccb,0,0,NaN,NaN,NaN,LLM-assisted curation (Claude) + pharmacology ...
4,apel,statin,0,0,NaN,NaN,NaN,LLM-assisted curation (Claude) + pharmacology ...


## 3. Perbaikan Data Sederhana

Contoh masalah data: `nasi-putih` sebelumnya berisi bahan seperti `tempe`, `kangkung`, dan `kecap manis`, padahal kelasnya adalah nasi putih biasa. Di sini kita pastikan `nasi-putih` hanya berisi `beras` dan `air`.

Catatan: kalau kamu menemukan makanan lain yang salah, perbaiki di cell ini.

In [3]:
# Perbaikan manual yang jelas salah
food_to_ingredients["nasi-putih"] = ["beras", "air"]

# Simpan kembali food_to_ingredient_kb.json
food_kb["food_to_ingredients"] = food_to_ingredients
with open(FOOD_KB_PATH, "w", encoding="utf-8") as f:
    json.dump(food_kb, f, ensure_ascii=False, indent=4)

# Buat ulang reverse mapping ingredient_to_food_kb.json
ingredient_to_foods = defaultdict(list)
for food_name, ingredients in food_to_ingredients.items():
    for ingredient in ingredients:
        ingredient_to_foods[ingredient].append(food_name)

reverse_kb = {
    "metadata": {
        "version": food_kb.get("metadata", {}).get("version", "1.0"),
        "total_ingredients": len(ingredient_to_foods),
        "description": "Reverse mapping ingredient ke daftar makanan.",
    },
    "ingredient_to_foods": {
        ingredient: sorted(foods)
        for ingredient, foods in sorted(ingredient_to_foods.items())
    },
}

with open(REVERSE_KB_PATH, "w", encoding="utf-8") as f:
    json.dump(reverse_kb, f, ensure_ascii=False, indent=4)

print("Ingredient nasi-putih:", food_to_ingredients["nasi-putih"])
print("Reverse KB tersimpan:", REVERSE_KB_PATH)

Ingredient nasi-putih: ['beras', 'air']
Reverse KB tersimpan: d:\tugas\coding camp 2026\capstone project\ai engineer\data\ingredient_to_food_kb.json


## 4. Definisi Risk Category

Target asli adalah severity skala 0-5:

- `0`: aman
- `1-2`: ringan
- `3`: sedang
- `4-5`: tinggi

In [4]:
def severity_to_risk(severity):
    if severity == 0:
        return "aman"
    if severity <= 2:
        return "ringan"
    if severity <= 3:
        return "sedang"
    return "tinggi"


def pred_to_risk(severity):
    if severity < 1.0:
        return "aman"
    if severity < 2.5:
        return "ringan"
    if severity < 3.5:
        return "sedang"
    return "tinggi"


df["risk_category"] = df["severity"].apply(severity_to_risk)

print("Distribusi severity:")
display(df["severity"].value_counts().sort_index().rename("count").to_frame())

print("Distribusi risk category:")
display(df["risk_category"].value_counts().rename("count").to_frame())

Distribusi severity:


,count
severity,
0,629
1,15
2,15
3,101
4,87
5,7


Distribusi risk category:


,count
risk_category,
aman,629
sedang,101
tinggi,94
ringan,30


## 5. Feature Engineering

Kita ubah daftar bahan makanan menjadi fitur 0/1.

Contoh:

- kalau makanan mengandung `kangkung`, maka fitur `kw_kangkung = 1`
- kalau tidak mengandung `kangkung`, maka `kw_kangkung = 0`

Fitur model:

- `drug_category`
- keyword bahan aktif seperti `kunyit`, `susu`, `tempe`, `gula pasir`, dll.

In [5]:
ACTIVE_KEYWORDS = [
    "bawang putih", "jahe", "kunyit", "lengkuas",
    "kangkung", "bayam", "kemangi", "daun bawang",
    "gula pasir", "gula merah", "gula aren", "kental manis", "madu",
    "pisang", "kentang",
    "santan",
    "jeruk",
    "susu",
    "kecap", "petis", "terasi",
    "tempe", "tahu",
    "cuka", "kopi",
]


def get_ingredient_features(food_name):
    ingredients = food_to_ingredients.get(food_name, [])
    features = []
    for keyword in ACTIVE_KEYWORDS:
        found = any(keyword.lower() in ingredient.lower() for ingredient in ingredients)
        features.append(1.0 if found else 0.0)
    return features


def build_feature_frame(dataframe):
    rows = []
    for _, row in dataframe.iterrows():
        food_name = row["food_class"]
        feature_values = get_ingredient_features(food_name)

        feature_row = {
            "food_class": food_name,
            "drug_category": row["drug_category"],
        }
        for keyword, value in zip(ACTIVE_KEYWORDS, feature_values):
            feature_row[f"kw_{keyword}"] = value
        rows.append(feature_row)

    return pd.DataFrame(rows)


X = build_feature_frame(df)
y = df["severity"].astype(float).to_numpy()
risk_labels = df["risk_category"].to_numpy()
keyword_columns = [f"kw_{keyword}" for keyword in ACTIVE_KEYWORDS]

print("Shape fitur:", X.shape)
display(X.head())

Shape fitur: (854, 27)


,food_class,drug_category,kw_bawang putih,kw_jahe,kw_kunyit,kw_lengkuas,kw_kangkung,kw_bayam,kw_kemangi,kw_daun bawang,...,kw_santan,kw_jeruk,kw_susu,kw_kecap,kw_petis,kw_terasi,kw_tempe,kw_tahu,kw_cuka,kw_kopi
0,apel,antikoagulan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,apel,antidiabetes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,apel,ace_arb,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,apel,ccb,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,apel,statin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 6. Build Model

Model yang dipakai: **Extra Trees Regressor**.

Kenapa regressor? Karena target yang diprediksi adalah severity numerik `0-5`. Setelah severity diprediksi, nilainya dikonversi menjadi risk category.

In [6]:
def make_model_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ("drug_category", OneHotEncoder(handle_unknown="ignore"), ["drug_category"]),
            ("ingredient_features", "passthrough", keyword_columns),
        ],
        remainder="drop",
    )

    model = ExtraTreesRegressor(
        n_estimators=400,
        random_state=42,
        min_samples_leaf=1,
        n_jobs=-1,
    )

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])

    return pipeline

## 7. Cross Validation

Evaluasi memakai 5-fold Stratified K-Fold berdasarkan risk category agar distribusi `aman`, `ringan`, `sedang`, dan `tinggi` tetap seimbang di tiap fold.

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
all_true_risk = []
all_pred_risk = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, risk_labels), start=1):
    model = make_model_pipeline()
    model.fit(X.iloc[train_idx], y[train_idx])

    pred = model.predict(X.iloc[val_idx])
    pred = np.clip(pred, 0.0, 5.0)

    true_severity = y[val_idx]
    true_risk = risk_labels[val_idx]
    pred_risk = np.array([pred_to_risk(value) for value in pred])

    mae = mean_absolute_error(true_severity, pred)
    rmse = np.sqrt(mean_squared_error(true_severity, pred))
    risk_acc = accuracy_score(true_risk, pred_risk)

    fold_results.append({
        "fold": fold,
        "mae": mae,
        "rmse": rmse,
        "risk_accuracy": risk_acc,
    })

    all_true_risk.extend(true_risk.tolist())
    all_pred_risk.extend(pred_risk.tolist())

    print(f"Fold {fold}: MAE={mae:.4f} | RMSE={rmse:.4f} | Risk Accuracy={risk_acc:.4f}")

cv_results = pd.DataFrame(fold_results)

print("\nRingkasan Cross Validation")
print(f"MAE           : {cv_results['mae'].mean():.4f} (+/- {cv_results['mae'].std():.4f})")
print(f"RMSE          : {cv_results['rmse'].mean():.4f} (+/- {cv_results['rmse'].std():.4f})")
print(f"Risk Accuracy : {cv_results['risk_accuracy'].mean() * 100:.2f}% (+/- {cv_results['risk_accuracy'].std() * 100:.2f}%)")

Fold 1: MAE=0.2052 | RMSE=0.7771 | Risk Accuracy=0.9064
Fold 2: MAE=0.1960 | RMSE=0.6989 | Risk Accuracy=0.9123
Fold 3: MAE=0.0893 | RMSE=0.4038 | Risk Accuracy=0.9474
Fold 4: MAE=0.1836 | RMSE=0.7222 | Risk Accuracy=0.9123
Fold 5: MAE=0.1460 | RMSE=0.5617 | Risk Accuracy=0.9235

Ringkasan Cross Validation
MAE           : 0.1640 (+/- 0.0475)
RMSE          : 0.6327 (+/- 0.1506)
Risk Accuracy : 92.04% (+/- 1.63%)


In [8]:
labels = ["aman", "ringan", "sedang", "tinggi"]

print("Classification Report:")
print(classification_report(all_true_risk, all_pred_risk, labels=labels, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(all_true_risk, all_pred_risk, labels=labels),
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
)

display(cm)

Classification Report:
              precision    recall  f1-score   support

        aman       0.95      0.97      0.96       629
      ringan       0.39      0.53      0.45        30
      sedang       0.95      0.82      0.88       101
      tinggi       0.93      0.85      0.89        94

    accuracy                           0.92       854
   macro avg       0.81      0.79      0.79       854
weighted avg       0.93      0.92      0.92       854



,pred_aman,pred_ringan,pred_sedang,pred_tinggi
true_aman,607,22,0,0
true_ringan,14,16,0,0
true_sedang,11,1,83,6
true_tinggi,8,2,4,80


## 8. Training Final dan Simpan Model

Setelah validasi selesai, model final dilatih menggunakan seluruh dataset dan disimpan sebagai file `.pkl`.

In [9]:
final_model = make_model_pipeline()
final_model.fit(X, y)

train_pred = np.clip(final_model.predict(X), 0.0, 5.0)
train_pred_risk = np.array([pred_to_risk(value) for value in train_pred])

train_mae = mean_absolute_error(y, train_pred)
train_rmse = np.sqrt(mean_squared_error(y, train_pred))
train_acc = accuracy_score(risk_labels, train_pred_risk)

print("Evaluasi pada seluruh training data")
print(f"MAE           : {train_mae:.4f}")
print(f"RMSE          : {train_rmse:.4f}")
print(f"Risk Accuracy : {train_acc * 100:.2f}%")

Evaluasi pada seluruh training data
MAE           : 0.0082
RMSE          : 0.1012
Risk Accuracy : 99.53%


In [10]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "model_type": "ExtraTreesRegressor",
    "pipeline": final_model,
    "active_keywords": ACTIVE_KEYWORDS,
    "keyword_columns": keyword_columns,
    "food_classes": sorted(df["food_class"].unique().tolist()),
    "severity_scale": "0-5",
    "metrics_cv": {
        "mae_mean": float(cv_results["mae"].mean()),
        "rmse_mean": float(cv_results["rmse"].mean()),
        "risk_accuracy_mean": float(cv_results["risk_accuracy"].mean()),
    },
    "metrics_train_full": {
        "mae": float(train_mae),
        "rmse": float(train_rmse),
        "risk_accuracy": float(train_acc),
    },
}

with open(MODEL_OUTPUT_PATH, "wb") as f:
    pickle.dump(model_bundle, f)

print("Model tersimpan di:", MODEL_OUTPUT_PATH)

Model tersimpan di: d:\tugas\coding camp 2026\capstone project\ai engineer\models\drug_interaction_tree_model.pkl


## 9. Mapping Nama Obat ke Kategori

Untuk demo sederhana, obat pasien dimapping ke kategori obat menggunakan keyword zat aktif.

In [11]:
DRUG_CATEGORIES = {
    "antikoagulan": ["WARFARIN", "CLOPIDOGREL", "HEPARIN", "RIVAROXABAN", "APIXABAN"],
    "antidiabetes": ["METFORMIN", "GLIBENCLAMIDE", "GLIMEPIRIDE", "INSULIN", "ACARBOSE"],
    "ace_arb": ["CAPTOPRIL", "ENALAPRIL", "LISINOPRIL", "LOSARTAN", "VALSARTAN", "CANDESARTAN"],
    "ccb": ["AMLODIPINE", "NIFEDIPINE", "DILTIAZEM", "VERAPAMIL"],
    "statin": ["SIMVASTATIN", "ATORVASTATIN", "ROSUVASTATIN", "PRAVASTATIN"],
    "antibiotik_tetrasiklin": ["DOXYCYCLINE", "TETRACYCLINE", "MINOCYCLINE"],
    "antibiotik_fluorokuinolon": ["CIPROFLOXACIN", "LEVOFLOXACIN", "MOXIFLOXACIN", "OFLOXACIN"],
    "maoi": ["SELEGILINE", "MOCLOBEMIDE", "LINEZOLID", "PHENELZINE"],
    "tiroid": ["LEVOTHYROXINE", "THYROXINE", "LEVOTIROKSIN"],
    "nsaid": ["IBUPROFEN", "DICLOFENAC", "DIKLOFENAK", "MELOXICAM", "KETOROLAC", "NAPROXEN"],
    "antikonvulsan": ["PHENYTOIN", "FENITOIN", "CARBAMAZEPINE", "VALPROIC"],
    "glikosida_jantung": ["DIGOXIN", "DIGOKSIN"],
    "xantin": ["THEOPHYLLINE", "TEOFILIN", "AMINOPHYLLINE"],
    "imunosupresan": ["CYCLOSPORINE", "TACROLIMUS", "SIROLIMUS", "EVEROLIMUS"],
}


def map_drug_to_categories(drug_name):
    drug_upper = drug_name.upper().strip()
    matched = []
    for category, keywords in DRUG_CATEGORIES.items():
        if any(keyword in drug_upper for keyword in keywords):
            matched.append(category)
    return matched


print("WARFARIN  ->", map_drug_to_categories("WARFARIN"))
print("METFORMIN ->", map_drug_to_categories("METFORMIN"))

# Tambahkan mapping obat ke artifact model agar bisa langsung dipakai app/API
model_bundle["drug_categories"] = {
    category: {
        "keywords": keywords,
        "mechanism": "Mekanisme interaksi mengikuti kategori obat dan bahan makanan aktif.",
    }
    for category, keywords in DRUG_CATEGORIES.items()
}

with open(MODEL_OUTPUT_PATH, "wb") as f:
    pickle.dump(model_bundle, f)

print("Artifact model diperbarui dengan drug category mapping.")

WARFARIN  -> ['antikoagulan']
METFORMIN -> ['antidiabetes']


## 10. Demo Rekomendasi Makanan

Untuk setiap makanan, sistem mengecek severity terhadap semua kategori obat pasien. Jika pasien minum lebih dari satu obat, skor makanan adalah severity terburuk dari semua obat.

In [12]:
def predict_food_category_severity(food_name, drug_category):
    sample = pd.DataFrame({
        "food_class": [food_name],
        "drug_category": [drug_category],
    })
    sample_features = build_feature_frame(sample)
    severity = float(final_model.predict(sample_features)[0])
    return float(np.clip(severity, 0.0, 5.0))


def recommend_foods(patient_medications, top_n=10):
    all_categories = []
    med_category_map = {}

    for medication in patient_medications:
        categories = map_drug_to_categories(medication)
        med_category_map[medication] = categories if categories else ["tidak ditemukan"]
        all_categories.extend(categories)

    all_categories = sorted(set(all_categories))

    if not all_categories:
        return pd.DataFrame({
            "food_name": sorted(df["food_class"].unique()),
            "severity_score": 0.0,
            "risk_level": "aman",
            "worst_category": None,
        }).head(top_n), med_category_map

    results = []
    for food_name in sorted(df["food_class"].unique()):
        scores = []
        for category in all_categories:
            severity = predict_food_category_severity(food_name, category)
            scores.append((category, severity))

        worst_category, max_severity = max(scores, key=lambda item: item[1])
        results.append({
            "food_name": food_name,
            "severity_score": round(max_severity, 2),
            "risk_level": pred_to_risk(max_severity),
            "worst_category": worst_category,
        })

    result_df = pd.DataFrame(results).sort_values("severity_score")
    return result_df.head(top_n), med_category_map


recommended, med_map = recommend_foods(["WARFARIN"], top_n=10)
print("Mapping obat:", med_map)
display(recommended)

Mapping obat: {'WARFARIN': ['antikoagulan']}


,food_name,severity_score,risk_level,worst_category
0,apel,0.0,aman,antikoagulan
7,bika-ambon,0.0,aman,antikoagulan
11,burger,0.0,aman,antikoagulan
15,es-dawet,0.0,aman,antikoagulan
13,cendol,0.0,aman,antikoagulan
14,donat,0.0,aman,antikoagulan
9,biskuit-choco-chips,0.0,aman,antikoagulan
23,kiwi,0.0,aman,antikoagulan
24,klappertart,0.0,aman,antikoagulan
25,kolak,0.0,aman,antikoagulan


In [13]:
recommended, med_map = recommend_foods(["METFORMIN", "SIMVASTATIN"], top_n=10)
print("Mapping obat:", med_map)
display(recommended)

Mapping obat: {'METFORMIN': ['antidiabetes'], 'SIMVASTATIN': ['statin']}


,food_name,severity_score,risk_level,worst_category
2,ayam-betutu,0.0,aman,antidiabetes
6,bakso,0.0,aman,antidiabetes
5,ayam-goreng-lengkuas,0.0,aman,antidiabetes
4,ayam-goreng,0.0,aman,antidiabetes
31,mie-goreng,0.0,aman,antidiabetes
21,kentang-goreng,0.0,aman,antidiabetes
58,tempe-goreng,0.0,aman,antidiabetes
57,telur-rebus,0.0,aman,antidiabetes
50,soto-banjar,0.0,aman,antidiabetes
55,tahu-telur,0.0,aman,antidiabetes


## 11. Cek Manual: Nasi Putih + Warfarin

Setelah perbaikan data, `nasi-putih` tidak lagi membawa bahan `tempe`, `kangkung`, atau `kecap`, sehingga tidak seharusnya menghasilkan warning untuk Warfarin.

In [14]:
severity_nasi_warfarin = predict_food_category_severity("nasi-putih", "antikoagulan")

print("Ingredient nasi-putih:", food_to_ingredients["nasi-putih"])
print("Prediksi severity nasi-putih + antikoagulan:", round(severity_nasi_warfarin, 2))
print("Risk level:", pred_to_risk(severity_nasi_warfarin))

Ingredient nasi-putih: ['beras', 'air']
Prediksi severity nasi-putih + antikoagulan: 0.0
Risk level: aman


## Kesimpulan

Model sederhana ini sudah cukup untuk baseline capstone:

- Metode: **Extra Trees Regressor**
- Input: kategori obat + fitur bahan makanan aktif
- Output: severity interaksi skala 0-5
- Rekomendasi: makanan diurutkan dari severity paling rendah

Hal yang paling perlu ditingkatkan berikutnya adalah kualitas data bahan makanan di `food_to_ingredient_kb.json`.